In [0]:
spark.conf.set(
    "***,
    "***"
)

In [0]:
bronze_path = "abfss://processed-data@adarshpmstorage01.dfs.core.windows.net/bronze"

silver_df = spark.read.format("delta").load(bronze_path)

display(silver_df)

Timestamp,Machine_ID,Assembly_Line_No,Hydraulic_Pressure_bar,Coolant_Pressure_bar,Air_System_Pressure_bar,Coolant_Temperature,Hydraulic_Oil_Temperature_C,Spindle_Bearing_Temperature_C,Spindle_Vibration_m,Tool_Vibration_m,Spindle_Speed_RPM,Voltage_volts,Torque_Nm,Cutting_kN,Downtime
11-09-2023 14:15:58,Makino-Synth-00419,Shopfloor-L8,109.7132,4.3031,6.5478,5.713,41.7116,35.3387,1.2634,45.1028,24141.3985,372.9501,36.5727,2.9406,No_Machine_Failure
07-10-2023 04:21:53,Makino-Synth-00128,Shopfloor-L5,169.165,5.6916,6.062,14.7895,49.4936,37.4462,1.4345,43.7645,20372.7976,350.2917,29.1389,3.4308,No_Machine_Failure
08-07-2024 02:55:44,Makino-Synth-00752,Shopfloor-L6,124.575,4.0546,6.5382,15.472,52.2523,35.5423,1.6844,44.4049,26646.9417,418.1107,33.7607,3.4114,No_Machine_Failure
14-01-2022 08:22:00,Makino-Synth-00427,Shopfloor-L3,97.727,5.0848,6.722,17.1958,51.474,30.8897,1.9138,32.2415,24250.5127,370.7215,33.0661,2.5327,No_Machine_Failure
20-08-2021 13:11:56,Makino-Synth-00368,Shopfloor-L1,187.2456,4.6532,7.0352,31.3214,45.4818,37.5983,1.8662,33.1116,18667.3906,371.4954,39.4073,3.3717,No_Machine_Failure
18-07-2024 10:12:49,Makino-Synth-00690,Shopfloor-L1,99.0087,4.7326,5.7988,11.0349,46.3041,40.6632,0.9843,29.7831,18204.9966,280.491,19.3545,2.2636,No_Machine_Failure
24-05-2022 20:08:18,Makino-Synth-00054,Shopfloor-L6,171.5344,4.3608,6.2769,8.71,48.9383,30.429,1.1943,29.9421,17116.5722,328.4784,23.9787,4.0146,No_Machine_Failure
24-08-2023 06:17:55,Makino-Synth-00281,Shopfloor-L4,139.5425,3.8487,6.1871,18.9495,43.9264,37.1567,1.9633,45.5573,19199.2563,367.8695,25.4303,2.4315,No_Machine_Failure
02-11-2022 03:22:35,Makino-Synth-00609,Shopfloor-L6,60.1034,5.0076,5.4998,41.4647,44.0031,35.5044,1.0871,38.4107,27153.544,372.8892,22.7355,3.0603,No_Machine_Failure
15-06-2024 22:00:47,Makino-Synth-00321,Shopfloor-L3,97.0399,5.1876,6.4815,4.7789,47.4432,38.7412,1.4872,21.0362,18330.165,375.5657,32.9491,3.813,No_Machine_Failure


In [0]:
silver_columns = [col.lower() for col in silver_df.columns]
silver_df = silver_df.toDF(*silver_columns)

silver_df.printSchema()

root
 |-- timestamp: string (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- assembly_line_no: string (nullable = true)
 |-- hydraulic_pressure_bar: double (nullable = true)
 |-- coolant_pressure_bar: double (nullable = true)
 |-- air_system_pressure_bar: double (nullable = true)
 |-- coolant_temperature: double (nullable = true)
 |-- hydraulic_oil_temperature_c: double (nullable = true)
 |-- spindle_bearing_temperature_c: double (nullable = true)
 |-- spindle_vibration_m: double (nullable = true)
 |-- tool_vibration_m: double (nullable = true)
 |-- spindle_speed_rpm: double (nullable = true)
 |-- voltage_volts: double (nullable = true)
 |-- torque_nm: double (nullable = true)
 |-- cutting_kn: double (nullable = true)
 |-- downtime: string (nullable = true)



In [0]:
silver_df = silver_df.dropna()
print("Rows after null removal:", silver_df.count())

Rows after null removal: 990045


In [0]:
silver_df = silver_df.dropDuplicates()
print("Rows after duplicate removal:", silver_df.count())

Rows after duplicate removal: 980828


In [0]:
from pyspark.sql.functions import col

silver_df = silver_df.filter(
    (col("hydraulic_pressure_bar") >= 0) &
    (col("coolant_pressure_bar") >= 0) &
    (col("air_system_pressure_bar") >= 0) &
    (col("coolant_temperature") >= 0) &
    (col("spindle_vibration_m") >= 0) &
    (col("tool_vibration_m") >= 0)
)

print("Rows after sanity filtering:", silver_df.count())

Rows after sanity filtering: 978401


In [0]:
silver_path = "abfss://processed-data@adarshpmstorage01.dfs.core.windows.net/silver"

silver_df.write.format("delta").mode("append").save(silver_path)

In [0]:
verify_silver = spark.read.format("delta").load(silver_path)

verify_silver.printSchema()
display(verify_silver)

root
 |-- timestamp: string (nullable = true)
 |-- machine_id: string (nullable = true)
 |-- assembly_line_no: string (nullable = true)
 |-- hydraulic_pressure_bar: double (nullable = true)
 |-- coolant_pressure_bar: double (nullable = true)
 |-- air_system_pressure_bar: double (nullable = true)
 |-- coolant_temperature: double (nullable = true)
 |-- hydraulic_oil_temperature_c: double (nullable = true)
 |-- spindle_bearing_temperature_c: double (nullable = true)
 |-- spindle_vibration_m: double (nullable = true)
 |-- tool_vibration_m: double (nullable = true)
 |-- spindle_speed_rpm: double (nullable = true)
 |-- voltage_volts: double (nullable = true)
 |-- torque_nm: double (nullable = true)
 |-- cutting_kn: double (nullable = true)
 |-- downtime: string (nullable = true)



timestamp,machine_id,assembly_line_no,hydraulic_pressure_bar,coolant_pressure_bar,air_system_pressure_bar,coolant_temperature,hydraulic_oil_temperature_c,spindle_bearing_temperature_c,spindle_vibration_m,tool_vibration_m,spindle_speed_rpm,voltage_volts,torque_nm,cutting_kn,downtime
11-01-2022 02:41:45,Makino-Synth-00044,Shopfloor-L2,116.177,3.8079,5.7951,20.7298,49.5432,38.0954,1.395,31.4333,16984.0576,311.8253,24.9367,3.1128,No_Machine_Failure
11-04-2023 16:19:39,Makino-Synth-00725,Shopfloor-L4,173.0649,4.396,6.7532,29.7766,50.7092,38.96,1.6492,36.2306,15562.4911,302.6555,27.2346,3.0019,No_Machine_Failure
14-06-2024 02:43:22,Makino-Synth-00199,Shopfloor-L8,122.3777,5.8749,6.6064,30.8863,55.3732,38.3555,1.3045,32.2337,20422.081,345.2439,19.4845,2.8946,No_Machine_Failure
25-09-2022 17:58:49,Makino-Synth-00102,Shopfloor-L8,97.7814,6.1863,6.6692,0.4788,49.2437,27.1742,1.5554,50.3614,15432.4345,343.0976,38.2867,3.7489,No_Machine_Failure
26-01-2022 00:36:35,Makino-Synth-00745,Shopfloor-L7,213.219,5.1724,6.1611,14.7283,50.0174,39.9527,1.2468,36.7141,20985.9923,403.8317,28.3275,3.754,No_Machine_Failure
09-05-2023 19:35:52,Makino-Synth-00273,Shopfloor-L2,135.3674,4.9236,6.3376,13.2566,48.1515,33.4285,1.4427,38.0111,17022.7449,351.5436,39.6228,3.6301,No_Machine_Failure
28-04-2023 22:02:52,Makino-Synth-00995,Shopfloor-L1,182.3217,4.8465,6.1841,23.8599,55.588,40.7393,1.6415,47.3887,15220.2683,287.7356,29.1318,2.8671,No_Machine_Failure
03-12-2022 21:29:55,Makino-Synth-00526,Shopfloor-L4,125.3049,5.6502,5.8703,15.5647,53.0704,38.6793,1.3259,24.4398,22184.2966,345.3991,32.3969,3.2774,No_Machine_Failure
07-09-2024 02:15:42,Makino-Synth-00128,Shopfloor-L5,187.732,7.2756,6.3316,29.1742,48.1836,41.6808,1.9206,49.475,16525.4057,352.8954,27.334,2.624,No_Machine_Failure
13-08-2021 08:47:17,Makino-Synth-00727,Shopfloor-L5,148.0183,5.6844,6.9665,11.196,52.2616,35.5553,0.7707,29.6017,19254.7196,374.4157,14.0456,3.4159,No_Machine_Failure


In [0]:
silver_df.select(
    "spindle_vibration_m",
    "tool_vibration_m",
    "spindle_bearing_temperature_c",
    "hydraulic_oil_temperature_c"
).show(10, False)

+-------------------+----------------+-----------------------------+---------------------------+
|spindle_vibration_m|tool_vibration_m|spindle_bearing_temperature_c|hydraulic_oil_temperature_c|
+-------------------+----------------+-----------------------------+---------------------------+
|0.6165             |18.5109         |33.2959                      |50.0764                    |
|2.1793             |26.5257         |33.6106                      |48.0392                    |
|1.3827             |27.7236         |34.3839                      |46.1614                    |
|1.0755             |42.2336         |37.9384                      |45.9407                    |
|1.0808             |26.6245         |38.7915                      |51.1359                    |
|0.9513             |34.8886         |36.0779                      |46.5506                    |
|1.0018             |25.885          |31.6175                      |46.7667                    |
|0.7537             |42.7375  